### 2. Building 5 predictive models

1. Ordinary Least Squares (OLS)
2. LASSO
3. Elastic Net
4. Random Forest
5. Gradient Boosting



In [43]:
# Import neccessary libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import ElasticNetCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [44]:
# Load the cleaned dataset
df = pd.read_csv("../data/processed/berlin_2025_Q3_clean.csv")
df.shape

(9264, 13)

After data cleaning and the removal of invalid or missing price observations, the final sample contains 9,264 listings. This reduction reflects standard preprocessing practices and ensures that the models are trained on valid and reliable observations.

In [45]:
# Target variable
y = df["log_price"]

# Features
X = df.drop(columns=["price", "log_price"])

# Encode categoricals
X = pd.get_dummies(X, drop_first=True)

# Ensure numeric + impute
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median())

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [46]:
# OLS Regression

# Convert everything explicitly to numpy float arrays
X_train_ols = sm.add_constant(X_train.to_numpy(dtype=float))
X_test_ols = sm.add_constant(X_test.to_numpy(dtype=float))

y_train_ols = y_train.to_numpy(dtype=float)
y_test_ols = y_test.to_numpy(dtype=float)

# Fit OLS
ols_model = sm.OLS(y_train_ols, X_train_ols).fit()

# Predict
y_pred_ols = ols_model.predict(X_test_ols)

# Metrics
rmse_ols = np.sqrt(mean_squared_error(y_test_ols, y_pred_ols))
r2_ols = r2_score(y_test_ols, y_pred_ols)

print("OLS RMSE:", rmse_ols)
print("OLS R²:", r2_ols)

OLS RMSE: 0.5113159735566617
OLS R²: 0.42075088712796793


In [47]:
# LASSO Regression model
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train_scaled, y_train)

y_pred_lasso = lasso.predict(X_test_scaled)

rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
r2_lasso = r2_score(y_test, y_pred_lasso)

print("LASSO RMSE:", rmse_lasso)
print("LASSO R²:", r2_lasso)

LASSO RMSE: 0.5113168405059069
LASSO R²: 0.4207489228630794


In [48]:
# Random forest model
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("RF RMSE:", rmse_rf)
print("RF R²:", r2_rf)

RF RMSE: 0.44561374669845516
RF R²: 0.5600494991693632


In [49]:
# Gradient boosting model
gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

gbr.fit(X_train, y_train)

y_pred_gbr = gbr.predict(X_test)

rmse_gbr = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
r2_gbr = r2_score(y_test, y_pred_gbr)

print("GB RMSE:", rmse_gbr)
print("GB R²:", r2_gbr)

GB RMSE: 0.44506308605027795
GB R²: 0.5611361520969675


In [50]:
# Elastic Net model
enet = ElasticNetCV(cv=5, random_state=42)
enet.fit(X_train_scaled, y_train)

y_pred_enet = enet.predict(X_test_scaled)

rmse_enet = np.sqrt(mean_squared_error(y_test, y_pred_enet))
r2_enet = r2_score(y_test, y_pred_enet)

print("Elastic Net RMSE:", rmse_enet)
print("Elastic Net R²:", r2_enet)

Elastic Net RMSE: 0.5113177694857838
Elastic Net R²: 0.42074681805040526


In [51]:
# Horserace table
results = pd.DataFrame({
    "Model": ["OLS", "LASSO", "Random Forest", "Gradient Boosting", "Elastic Net"],
    "RMSE": [rmse_ols, rmse_lasso, rmse_rf, rmse_gbr, rmse_enet],
    "R_squared": [r2_ols, r2_lasso, r2_rf, r2_gbr, r2_enet]
})

results

,Model,RMSE,R_squared
0,OLS,0.511316,0.420751
1,LASSO,0.511317,0.420749
2,Random Forest,0.445614,0.560049
3,Gradient Boosting,0.445063,0.561136
4,Elastic Net,0.511318,0.420747


The results show a clear separation between linear and tree-based models. OLS, LASSO, and Elastic Net all perform very similarly, with RMSE values around 0.47 and an R² of roughly 0.42. This indicates that linear specifications capture some of the variation in Airbnb prices, but leave a share unexplained.

The tree-based models perform  better. Random Forest reduces the prediction error and increases the explained variance, while Gradient Boosting delivers the strongest overall performance, achieving the lowest RMSE and the highest R². This suggests that nonlinearities and interaction effects play an important role in price formation and are better captured by more flexible models. Overall, the comparison indicates that while regularization does not  improve over OLS, machine-learning provides a gain in predictive accuracy.